The code below is calling a HUGGINGFACE model, the prompt needs to be improved more. Need to find a way for the model to be able to learn so that it does not have to repeat itself. Currently, the model is just outputting the prompt, but the prompt has to actually be passed into the model that is being called.

In [ ]:
!pip install faker
!pip install transformers
!pip install torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 20.5 MB/s eta 0:00:00


In [ ]:
# Import necessary libraries
import pandas as pd
import random
import numpy as np
from faker import Faker
from transformers import pipeline, MT5ForConditionalGeneration, AutoTokenizer
import time
from IPython.display import display
from datetime import datetime, timedelta
from google.colab import files
uploaded = files.upload()

import json
file_name = 'anonymized_ed discussion_discussion_data.json'

with open(file_name, 'r') as f:
    data = json.load(f)


# Initialize Hugging Face model pipeline for text generation with distilgpt2
#feedback_generator = pipeline("text-generation", model="distilgpt2")

# Initialize Faker to generate random names and other details
fake = Faker()

# Seed for reproducibility
random.seed(42)
np.random.seed(42)

# Set up the model and tokenizer
#model_name = "google/mt5-base"
model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = MT5ForConditionalGeneration.from_pretrained(model_name)

# 1. Generate Student Information
def generate_students(num_students=20):  # Generate 20 entries
    students = []
    for i in range(num_students):
        student_id = f"S{i+1:03}"  # Unique ID like S001, S002, etc.
        name = fake.name()
        major = random.choice(["Computer Science", "Mechanical Engineering", "Electrical Engineering", "Mathematics"])
        year = random.choice(["Freshman", "Sophomore"])
        students.append([student_id, name, major, year])
    return pd.DataFrame(students, columns=["StudentID", "Name", "Major", "Year"])

# 2. Generate Course Topics and Learning Modules
def generate_modules():
    modules = [
        ["M1", "Variables and Data Types", ["Integers", "Strings", "Booleans"], 1, "Understand basic data types and variables"],
        ["M2", "Loops and Conditionals", ["For Loops", "If Statements"], 2, "Use loops for repetition and conditions"],
        ["M3", "Functions", ["Defining Functions", "Parameters"], 2, "Define and use functions in Python"],
        ["M4", "Data Structures", ["Lists", "Dictionaries"], 2, "Use basic data structures"],
        ["M5", "File I/O", ["Reading Files", "Writing Files"], 1, "Read from and write to files in Python"]
    ]
    return pd.DataFrame(modules, columns=["ModuleID", "Topic", "Subtopics", "WeeksCovered", "LearningOutcomes"])

# 3. Generate Assignments and Exams
def generate_assignments(modules_df):
    assignments = []
    for _, module in modules_df.iterrows():
        assignment_id = f"A{module['ModuleID'][1]}"  # A1 for M1, A2 for M2, etc.
        assignment_topic = f"{module['Topic']} Practice"
        due_date = fake.date_between(start_date="-2y", end_date="today")
        max_score = 100
        average_score = random.randint(70, 90)
        assignments.append([assignment_id, module["ModuleID"], assignment_topic, due_date, max_score, average_score])
    return pd.DataFrame(assignments, columns=["AssignmentID", "ModuleID", "AssignmentTopic", "DueDate", "MaxScore", "AverageScore"])

# 4. Generate Student Performance Metrics
def generate_performance(students_df, modules_df):
    performance = []
    for _, student in students_df.iterrows():
        for _, module in modules_df.iterrows():
            assignment_score = random.randint(60, 100)
            exam_score = random.randint(60, 100)
            participation = f"{random.randint(8, 10)}/10"  # Assume participation score out of 10
            project_score = random.randint(70, 100)
            performance.append([student["StudentID"], module["ModuleID"], assignment_score, exam_score, participation, project_score])
    return pd.DataFrame(performance, columns=["StudentID", "ModuleID", "AssignmentScore", "ExamScore", "Participation", "ProjectScore"])

# 5. Generate Engagement and Effort Metrics
def generate_engagement(students_df, modules_df):
    engagement = []
    for _, student in students_df.iterrows():
        for _, module in modules_df.iterrows():
            time_spent = random.randint(3, 10)  # Time spent per week in hours
            practice_problems_solved = random.randint(5, 15)
            office_hours_visits = random.randint(0, 3)
            group_study_sessions = random.randint(0, 2)
            engagement.append([student["StudentID"], module["ModuleID"], time_spent, practice_problems_solved, office_hours_visits, group_study_sessions])
    return pd.DataFrame(engagement, columns=["StudentID", "ModuleID", "TimeSpent", "PracticeProblemsSolved", "OfficeHoursVisits", "GroupStudySessions"])



In [ ]:

def generate_content(prompt):
    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
    outputs = model.generate(
        **inputs,
        max_length=100,
        num_return_sequences=1,
        temperature=0.7,
        top_k=50,
        top_p=0.95,
        do_sample=True
    )
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Ensure the output is a coherent sentence
    if "<extra_id_0>" in generated_text:
        generated_text = generated_text.replace("<extra_id_0>", "").strip()

    return generated_text

In [ ]:
# 6. Generate Course Feedback using a Hugging Face model
def generate_feedback(students_df, modules_df, file_name):
    feedback = []
    topics = ["Clarification Request", "Conceptual Question", "Course Logistics"]
    for _, student in students_df.iterrows():
        for _, module in modules_df.iterrows():
            # Detailed prompt to generate realistic student feedback
            student_posts = []
            topic = random.choice(topics)
            f = file_name
            prompt = f"Generate a student post about {topic} with reference to a college intro to Computer Science class. It should be a question that needs an answer with a specific topic {topic}. This should be a full sentence and use the uploaded file {f} with the prompts that have the following tag: ""user": {"name": "<PERSON>", "email": "<EMAIL>", "role": "student"}""
            generated_text = generate_content(prompt)

            # Generate feedback text using the Hugging Face model
            # try:
            #     response = feedback_generator(prompt, max_new_tokens=50, truncation=True, num_return_sequences=1)
            #     generated_feedback = response[0]["generated_text"]  # Capturing model's output
            # except Exception as e:
            #     generated_feedback = f"Error generating feedback: {str(e)}"
            confidence_level = random.randint(1, 5)
            feedback.append([student["StudentID"], module["ModuleID"], generated_text, confidence_level])

    return pd.DataFrame(feedback, columns=["StudentID", "ModuleID", "StudentFeedback", "ConfidenceLevel"])

# Generate the datasets
students_df = generate_students(num_students=20)  # Generate 20 entries
modules_df = generate_modules()
assignments_df = generate_assignments(modules_df)
performance_df = generate_performance(students_df, modules_df)
engagement_df = generate_engagement(students_df, modules_df)
feedback_df = generate_feedback(students_df, modules_df)

# Display the datasets
print("Students DataFrame:")
display(students_df)

print("\nModules DataFrame:")
display(modules_df)

print("\nAssignments DataFrame:")
display(assignments_df)

print("\nPerformance DataFrame:")
display(performance_df)

print("\nEngagement DataFrame:")
display(engagement_df)

print("\nFeedback DataFrame:")
display(feedback_df)

# Save to CSV files for further use
students_df.to_csv("students.csv", index=False)
modules_df.to_csv("modules.csv", index=False)
assignments_df.to_csv("assignments.csv", index=False)
performance_df.to_csv("performance.csv", index=False)
engagement_df.to_csv("engagement.csv", index=False)
feedback_df.to_csv("feedback.csv", index=False)


SyntaxError: invalid syntax (<ipython-input-8-47a96d0471b1>, line 11)